<a href="https://colab.research.google.com/github/lucas-j-zheng/ml-practice-from-scratch/blob/main/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import RobertaModel, RobertaTokenizerFast
from sklearn.metrics import accuracy_score, matthews_corrcoef
from scipy.stats import pearsonr
import random
from datasets import load_dataset


import math


In [19]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
def set_seed(s=42):
  random.seed(s)
  np.random_seed(s)
  torch.manual_seed(s)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(s)

TASK_CONFIG = {
    'mrpc':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 2, 'is_regression': False},
    'cola':  {'cols': ('sentence',  None),         'num_outputs': 2, 'is_regression': False},
    'stsb':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 1, 'is_regression': True},
}

In [20]:
MODEL_NAME = 'roberta-base'
HIDDEN = 768

## Data Loading



In [21]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)


In [26]:
class GLUEDataset(torch.utils.data.Dataset):
  def __init__(self, texts_a, texts_b, labels, tokenizer, is_regression):
    self.texts_a = texts_a
    self.texts_b = texts_b
    self.labels = labels
    self.tokenizer = tokenizer
    self.is_regression = is_regression

  def __len__(self):
    return len(self.labels)


  def __getitem__(self, i):
    # tokenize and return one example
    if self.texts_b is None:
      encoded = self.tokenizer(
        self.texts_a[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    else:
      encoded = self.tokenizer(
        self.texts_a[i], self.texts_b[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    label_dtype = torch.float if self.is_regression else torch.long
    label = torch.tensor(self.labels[i], dtype=label_dtype)

    # squeeze for additional dimension from turning into pt (batch dim)
    return {
        'input_ids': encoded['input_ids'].squeeze(0),
        'attention_mask': encoded['attention_mask'].squeeze(0),
        'labels': label,
    }





In [27]:
def build_dataloaders(task, batch_size=32):
  cfg = TASK_CONFIG[task]
  col_a, col_b = cfg['cols']
  is_regression = cfg['is_regression']
  raw = load_dataset('glue', task)

  def make_split(split):
    a = raw[split][col_a]
    b = raw[split][col_b] if col_b is not None else None
    y = raw[split]['label']
    return GLUEDataset(a, b, y, tokenizer, is_regression)

  train = make_split('train')
  val = make_split('validation')
  test = make_split('test')

  train_dl = DataLoader(train, batch_size=batch_size, shuffle=True)
  val_dl = DataLoader(val, batch_size=batch_size, shuffle=False)

  return train_dl, val_dl

# Model Finetuning (FULL)

In [28]:
class RobertaClassifier(nn.Module):
  def __init__(self, base, num_outputs):
    super().__init__()
    self.base = base
    self.dropout = nn.Dropout(0.1)
    self.head = nn.Linear(HIDDEN, num_outputs)

  def forward(self, input_ids, attention_mask):
    # forward pass through the base model
    out = self.base(input_ids = input_ids, attention_mask = attention_mask)

    #Take the position 0 <s> token because that's the summary token
    # for bert, lora, summarizes everything
    # removes rest of the tokens, Shape (B, 768)
    CLS = out.last_hidden_state[:, 0, :]
    a = self.dropout(CLS)
    b = self.head(a)
    return b






Testing Roberta Classifier


In [29]:
base = RobertaModel.from_pretrained('roberta-base')
model = RobertaClassifier(base, num_outputs=2)

ids = torch.randint(0, 1000, (4, 128))
mask = torch.ones_like(ids)
out = model(ids, mask)
print(out.shape)   # expect (4, 2) due to (4, num_outputs)
print(out.dtype)   # expect torch.float32

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([4, 2])
torch.float32


In [30]:
def evaluate(model, dataloader, task, device):
  model.eval()
  cfg = TASK_CONFIG[task]
  is_regression = cfg['is_regression']
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch in dataloader:
      batch = {k: v.to(device) for k,v in batch.items()}
      logits = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'])

      if is_regression:
        preds = logits.squeeze(-1)

      else:
        preds = logits.argmax(dim = -1)

      all_preds.append(preds.cpu().numpy())
      all_labels.append(batch['labels'].cpu().numpy())

  all_preds = np.concatenate(all_preds)
  all_labels = np.concatenate(all_labels)

  if task == 'mrpc':
    return accuracy_score(all_labels, all_preds)
  elif task == 'cola':
    return matthews_corrcoef(all_labels, all_preds)
  elif task == 'stsb':
    return pearsonr(all_preds, all_labels)[0]




In [32]:
def train(task, use_lora=False, epochs=3, lr=2e-5, batch_size=32):
  #set seed first
  set_seed(42)
  train_dl, val_dl = build_dataloaders(task, batch_size)
  cfg = TASK_CONFIG[task]
  is_regression = cfg['is_regression']

  base = RobertaModel.from_pretrained('roberta-base')
  model = RobertaClassifier(base, num_outputs=cfg['num_outputs'])
  model.to(DEVICE)

  #need loss

  loss_fn = nn.MSELoss() if is_regression else nn.CrossEntropyLoss()

  #only optimizer for trainable params!

  optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=lr, weight_decay=0.01
                                )

  best = -1e9

  for epoch in range(epochs):
    model.train()

    for batch in train_dl:
      # a. move to device
      batch = {k: v.to(DEVICE) for k, v in batch.items()}

      # b. forward
      logits = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'])

      # c. compute loss (branch on regression)
      if is_regression:
        loss = loss_fn(logits.squeeze(-1), batch['labels'])
      else:
        loss = loss_fn(logits, batch['labels'])

      # d. zero grads
      optimizer.zero_grad()

      # e. backward
      loss.backward()

      # f. step
      optimizer.step()

    val_metric = evaluate(model, val_dl, task, DEVICE)
    best = max(best, val_metric)
    print(f'  epoch {epoch+1}: val={val_metric:.4f}')

  return best
